# 1.0 — Setup & Imports

Import all required standard, scientific, and deep-learning libraries.

In [1]:
# ── Kaggle environment setup ──────────────────────────────────────────────────
import os
import matplotlib
matplotlib.use('Agg')  # headless – no display on Kaggle

import tensorflow as tf

# Allow GPU memory growth (avoids OOM on shared Kaggle GPU)
gpus = tf.config.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

print('GPU devices:', gpus)
print('TF version :', tf.__version__)

GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
TF version : 2.20.0


In [2]:
# ── Standard library ────────────────────────────────────────────────────────────
import glob
import json
import time
import random
from pathlib import Path

from scipy.io import loadmat

# ── Scientific / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ──────────────────────────────────────────────────────────────
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# ── Deep learning ──────────────────────────────────────────────────────────────
from tensorflow import keras
from tensorflow.keras import layers

# ── ML utilities ───────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, confusion_matrix,
    classification_report, f1_score, log_loss
)

# ── Excel export ───────────────────────────────────────────────────────────────
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.drawing.image import Image as XLImage

sns.set_style('whitegrid')
print('TensorFlow:', tf.__version__)

TensorFlow: 2.20.0


# 2.0 — Configuration

All paths, seeds, dataset geometry, training hyperparameters, and per-architecture configs in one place. **Edit only this section between experiments.**

In [3]:
# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Directory layout & environment detection ──────────────────────────────────
IN_KAGGLE = Path('/kaggle').exists()

_data_matches = (
    glob.glob('/kaggle/input/**/k001.csv', recursive=True)
    or glob.glob('/kaggle/input/**/data.csv', recursive=True)
    or glob.glob('Classification/data/data.csv')
    or glob.glob('../data/data.csv')
    or glob.glob('../../Classification/data/data.csv')
    or glob.glob('/content/drive/My Drive/Classification/data/data.csv')
    or glob.glob('data.csv')
)
_ref_matches = (
    glob.glob('/kaggle/input/**/k002.csv', recursive=True)
    or glob.glob('/kaggle/input/**/ref.csv', recursive=True)
    or glob.glob('Classification/data/ref.csv')
    or glob.glob('../data/ref.csv')
    or glob.glob('../../Classification/data/ref.csv')
    or glob.glob('/content/drive/My Drive/Classification/data/ref.csv')
    or glob.glob('ref.csv')
)

DATA_FILE  = Path(_data_matches[0])  if _data_matches else None
LABEL_FILE = Path(_ref_matches[0])   if _ref_matches  else None

# ── Output directories ────────────────────────────────────────────────────────
if IN_KAGGLE:
    MODEL_DIR   = Path('/kaggle/working/credit/models')
    RESULTS_DIR = Path('/kaggle/working/credit/results')
else:
    MODEL_DIR   = Path('./models')
    RESULTS_DIR = Path('./results')

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = RESULTS_DIR / 'training_plots'
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Resolve ENSEMBLE_DIR (ensemble member .keras files) ────────────────────────
candidates = [
    Path("/kaggle/input/cuq6-ensemble-k04"),
    Path("/kaggle/input/notebooks/daneshselwal/cuq6-ensemble-k04"),
    Path("/kaggle/input/dias-ensemble-models"),
    Path("/kaggle/input/dias-ensemble-models/ensembles"),
    Path("/kaggle/input/cuq-ensemble-models"),
    Path("/kaggle/input"),
    Path("../ensemble/models"),
    Path("Classification/ensemble/models"),
    Path("./models"),
]

ENSEMBLE_DIR = None
for c in candidates:
    if c.exists():
        keras_files = list(c.rglob("*_ens_*_final.keras"))
        if keras_files:
            ENSEMBLE_DIR = c
            print(f"ENSEMBLE_DIR resolved to: {ENSEMBLE_DIR} ({len(keras_files)} .keras files)")
            break
        else:
            keras_files_any = list(c.rglob("*.keras"))
            if keras_files_any:
                ENSEMBLE_DIR = c
                print(f"ENSEMBLE_DIR resolved to: {ENSEMBLE_DIR} (contains {len(keras_files_any)} .keras files)")
                break

if ENSEMBLE_DIR is None:
    all_ens = sorted(Path("/kaggle/input").rglob("*_ens_*_final.keras")) if Path("/kaggle/input").exists() else []
    all_kN = sorted(Path("/kaggle/input").rglob("k0[12][0-9].keras")) if Path("/kaggle/input").exists() else []
    all_found = all_ens + all_kN
    if all_found:
        ENSEMBLE_DIR = all_found[0].parent
        print(f"ENSEMBLE_DIR resolved via rglob to: {ENSEMBLE_DIR} ({len(all_found)} files)")
    else:
        ENSEMBLE_DIR = Path("./models")
        print(f"ENSEMBLE_DIR default: {ENSEMBLE_DIR}")

# ── Dataset geometry (6 Bands) ────────────────────────────────────────────────
H, W, B     = 330, 307, 6
PATCH_SIZE  = 9
INNER_PATCH = 3

# ── Split ratios ──────────────────────────────────────────────────────────────
TRAIN_PERCENT        = 0.75
VAL_SPLIT_FROM_TRAIN = 0.20

# ── Training hyperparameters ──────────────────────────────────────────────────
BATCH_SIZE    = 128
EPOCHS        = 100
LEARNING_RATE = 3e-4
DROPOUT_RATE  = 0.25

TRAIN_CFG = {
    'label_smoothing': 0.05,
    'weight_decay':    1e-4,
    'clipnorm':        1.0,
    'cosine_alpha':    0.05,
}

# ── Architecture configs ──────────────────────────────────────────────────────
ALEXNET_CFG = {
    'conv_filters': [96, 256, 384, 384, 256],
    'dense_units':  [4096, 1024, 256, 32],
}
GFNET_CFG = {
    'hidden_dim': 512,
    'num_blocks': 5,
    'mlp_ratio':  4,
}
VIT_CFG = {
    'projection_dim':     256,
    'num_heads':          4,
    'transformer_layers': 12,
    'mlp_multiplier':     2,
    'head_units':         [512, 256, 128, 64],
}

# ── AlexNet legacy recipe ─────────────────────────────────────────────────────
ALEXNET_LEGACY_SPLIT_SEED    = 10
ALEXNET_LEGACY_TRAIN_PERCENT = 0.75
ALEXNET_LR_START = 0.01
ALEXNET_LR_MAX   = 0.02
ALEXNET_LR_MIN   = 0.005

# ── Sanity print ──────────────────────────────────────────────────────────────
print('Data   :', DATA_FILE)
print('Labels :', LABEL_FILE)
print('Ensembles:', ENSEMBLE_DIR)
print('Models :', MODEL_DIR)
print('Results:', RESULTS_DIR)


ENSEMBLE_DIR resolved to: /kaggle/input/cuq6-ensemble-k04 (15 .keras files)
Data   : /kaggle/input/dias-6-bands-data/k001.csv
Labels : /kaggle/input/dias-6-bands-data/k002.csv
Ensembles: /kaggle/input/cuq6-ensemble-k04
Models : /kaggle/working/credit/models
Results: /kaggle/working/credit/results


# 3.0 — Data Loading & Patch Extraction

Reads the hyperspectral/multispectral scene from .mat files, applies a robust per-band percentile-clip normalisation, extracts fixed-size spatial patches centred on every labelled pixel, and builds both the standard and AlexNet-legacy train/test splits.

## 3.1 — Helper Functions

In [4]:
def load_multispectral_6band(data_path, label_path, height, width, bands):
    """Read CSV rasters, reshape to (H, W, B), and apply per-band min-max normalisation."""
    x = pd.read_csv(data_path).to_numpy(dtype=np.float32).reshape(height, width, bands)
    y = pd.read_csv(label_path).to_numpy(dtype=np.int32).reshape(height, width)

    x_norm = np.empty_like(x, dtype=np.float32)
    for b in range(bands):
        band = x[:, :, b]
        b_min, b_max = np.min(band), np.max(band)
        x_norm[:, :, b] = (band - b_min) / max(b_max - b_min, 1e-8)
    return x_norm, y


def extract_labeled_patches(x, y, patch_size=9):
    """Extract spatial patches centred on every labelled (y > 0) pixel."""
    pad   = patch_size // 2
    x_pad = np.pad(x, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

    coords  = np.argwhere(y > 0)
    patches = np.empty((len(coords), patch_size, patch_size, x.shape[-1]), dtype=np.float32)
    labels  = np.empty((len(coords),), dtype=np.int32)

    for i, (r, c) in enumerate(coords):
        patches[i] = x_pad[r:r + patch_size, c:c + patch_size, :]
        labels[i]  = int(y[r, c]) - 1   # 1-indexed → 0-indexed

    return patches, labels, coords


## 3.2 — Load Image & Extract Patches

In [5]:
print('Loading multispectral image...')
x_img, y_img = load_multispectral_6band(DATA_FILE, LABEL_FILE, H, W, B)

print('Extracting patches...')
X, y, coords = extract_labeled_patches(x_img, y_img, PATCH_SIZE)

num_classes = int(np.unique(y).size)
input_shape = (PATCH_SIZE, PATCH_SIZE, B)

print(f'Scene geometry : H={H}, W={W}, B={B}')
print(f'Image          : {x_img.shape}  |  Labels: {y_img.shape}')
print(f'Patches shape  : {X.shape}  |  Classes: {num_classes}')


Loading multispectral image...
Extracting patches...
Scene geometry : H=330, W=307, B=6
Image          : (330, 307, 6)  |  Labels: (330, 307)
Patches shape  : (17239, 9, 9, 6)  |  Classes: 7


## 3.3 — Train / Validation / Test Split

Two independent splits are created: a **standard split** (shared by GFNet and ViT) and an **AlexNet legacy split** (separate seed to preserve reproducibility of uncertainty recovery).

In [6]:
# ── Standard split (GFNet & ViT) ──────────────────────────────────────────────
x_train_full, x_test, y_train_full, y_test = train_test_split(
    X, y, train_size=TRAIN_PERCENT, random_state=SEED, stratify=y
)
x_train, x_val, y_train, y_val = train_test_split(
    x_train_full, y_train_full,
    test_size=VAL_SPLIT_FROM_TRAIN, random_state=SEED, stratify=y_train_full
)

y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_val_cat   = keras.utils.to_categorical(y_val,   num_classes)
y_test_cat  = keras.utils.to_categorical(y_test,  num_classes)

print(f'Train : {x_train.shape}  Val : {x_val.shape}  Test : {x_test.shape}')

# ── AlexNet legacy split (separate seed for uncertainty recovery) ──────────────
x_train_alex, x_test_alex, y_train_alex, y_test_alex = train_test_split(
    X, y,
    train_size=ALEXNET_LEGACY_TRAIN_PERCENT,
    random_state=ALEXNET_LEGACY_SPLIT_SEED,
    stratify=y
)
y_train_alex_cat = keras.utils.to_categorical(y_train_alex, num_classes)
y_test_alex_cat  = keras.utils.to_categorical(y_test_alex,  num_classes)

print(f'AlexNet  Train : {x_train_alex.shape}  Test : {x_test_alex.shape}')

Train : (10343, 9, 9, 6)  Val : (2586, 9, 9, 6)  Test : (4310, 9, 9, 6)
AlexNet  Train : (12929, 9, 9, 6)  Test : (4310, 9, 9, 6)


# 4.0 — Model Definitions

Defines all three single-head base architectures (AlexNet CNN, GFNet, ViT with U-Net skip connections) plus the shared custom Keras layers used by GFNet and ViT.

## 4.1 — Shared Custom Keras Layers

Four serialisable custom layers: patch extraction, positional encoding (with and without CLS token), and the frequency-domain Global Filter layer used in GFNet.

In [7]:
@tf.keras.utils.register_keras_serializable()
class PatchExtractor(layers.Layer):
    """Extract non-overlapping patches from a spatial image tensor."""

    def __init__(self, patch_size=3, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding='VALID',
        )
        batch       = tf.shape(images)[0]
        num_patches = tf.shape(patches)[1] * tf.shape(patches)[2]
        patch_dim   = tf.shape(patches)[-1]
        return tf.reshape(patches, [batch, num_patches, patch_dim])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'patch_size': self.patch_size})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchPositionEncoder(layers.Layer):
    """Linear projection + learnable position embedding (no CLS token)."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=projection_dim
        )

    def call(self, patches):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patches) + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


@tf.keras.utils.register_keras_serializable()
class PatchEncoderWithCLS(layers.Layer):
    """Linear projection + learnable position embedding WITH a prepended CLS token."""

    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches    = num_patches
        self.projection_dim = projection_dim
        self.projection         = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches + 1, output_dim=projection_dim
        )

    def build(self, input_shape):
        self.cls_token = self.add_weight(
            name='cls_token', shape=(1, 1, self.projection_dim),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, patches):
        batch      = tf.shape(patches)[0]
        patch_proj = self.projection(patches)
        cls_tokens = tf.repeat(self.cls_token, repeats=batch, axis=0)
        x          = tf.concat([cls_tokens, patch_proj], axis=1)
        positions  = tf.range(start=0, limit=self.num_patches + 1, delta=1)
        return x + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'num_patches': self.num_patches, 'projection_dim': self.projection_dim})
        return cfg


@tf.keras.utils.register_keras_serializable()
class GlobalFilterLayer(layers.Layer):
    """Learnable complex-valued filter applied in the 2-D frequency domain."""

    def __init__(self, token_side, **kwargs):
        super().__init__(**kwargs)
        self.token_side = token_side

    def build(self, input_shape):
        channels = int(input_shape[-1])
        self.w_real = self.add_weight(
            name='w_real', shape=(self.token_side, self.token_side, channels),
            initializer='glorot_uniform', trainable=True
        )
        self.w_imag = self.add_weight(
            name='w_imag', shape=(self.token_side, self.token_side, channels),
            initializer='zeros', trainable=True
        )
        super().build(input_shape)

    def call(self, x):
        batch    = tf.shape(x)[0]
        channels = tf.shape(x)[-1]
        x_2d       = tf.reshape(x, [batch, self.token_side, self.token_side, channels])
        x_fft      = tf.signal.fft2d(tf.cast(x_2d, tf.complex64))
        w_complex  = tf.complex(self.w_real, self.w_imag)
        x_filtered = x_fft * w_complex
        x_spatial  = tf.math.real(tf.signal.ifft2d(x_filtered))
        return tf.reshape(x_spatial, [batch, self.token_side * self.token_side, channels])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'token_side': self.token_side})
        return cfg

## 4.2 — AlexNet CNN

In [8]:
def build_alexnet(input_shape, num_classes, dropout_rate=0.25, cfg=None):
    """Build a single-head AlexNet-style CNN with configurable conv and dense units."""
    cfg = cfg or ALEXNET_CFG

    inputs = keras.Input(shape=input_shape)
    x = inputs

    for i, filters in enumerate(cfg['conv_filters'], start=1):
        x = layers.Conv2D(filters, (3, 3), activation='relu',
                          padding='same', name=f'alex_conv_{i}')(x)

    x = layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2),
                            padding='same', name='alex_pool')(x)
    x = layers.Flatten(name='alex_flatten')(x)

    fc_names   = ['alex_fc1', 'alex_fc2', 'alex_fc3', 'alex_fc4']
    drop_names = ['TRAIN_DROPOUT_1', 'TRAIN_DROPOUT_2', 'TRAIN_DROPOUT_3', None]

    for units, fc_name, drop_name in zip(cfg['dense_units'], fc_names, drop_names):
        x = layers.Dense(units, activation='relu', name=fc_name)(x)
        if drop_name:
            x = layers.Dropout(dropout_rate, name=drop_name)(x)

    outputs = layers.Dense(num_classes, activation='softmax', name='alex_logits')(x)
    return keras.Model(inputs, outputs, name='AlexNet_SingleHead')

## 4.3 — Global Filter Network (GFNet)

In [9]:
def gf_block(x, token_side, dim, mlp_ratio=4, dropout_rate=0.25, name_prefix='gf'):
    """One GFNet block: Global Filter → LayerNorm → MLP → residual add."""
    y = layers.LayerNormalization(name=f'{name_prefix}_ln1')(x)
    y = GlobalFilterLayer(token_side, name=f'{name_prefix}_gfilter')(y)
    y = layers.LayerNormalization(name=f'{name_prefix}_ln2')(y)
    y = layers.Dense(dim * mlp_ratio, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp1')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop1')(y)
    y = layers.Dense(dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp2')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop2')(y)
    return layers.Add(name=f'{name_prefix}_add')([x, y])


def build_gfnet(input_shape, num_classes, inner_patch=3,
                hidden_dim=512, num_blocks=5, mlp_ratio=4, dropout_rate=0.25):
    """Build a single-head Global Filter Network over tokenised image patches."""
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)
    token_side  = int(np.sqrt(num_patches))

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name='gf_patch_extractor')(inputs)
    x = PatchPositionEncoder(num_patches, hidden_dim, name='gf_patch_encoder')(x)
    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_1')(x)

    for i in range(num_blocks):
        x = gf_block(x, token_side, hidden_dim, mlp_ratio,
                     dropout_rate, name_prefix=f'gf_block_{i+1}')

    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_2')(x)
    x = layers.LayerNormalization(name='gf_final_ln')(x)
    x = layers.GlobalAveragePooling1D(name='gf_gap')(x)
    x = layers.Flatten(name='gf_flatten')(x)
    x = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_3')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='gf_logits')(x)

    return keras.Model(inputs, outputs, name='GFNet_SingleHead')

## 4.4 — Vision Transformer with U-Net Skip Connections

In [10]:
def transformer_block(x, num_heads, projection_dim, mlp_dim, dropout_rate, name_prefix):
    """Standard pre-LN transformer block: Multi-Head Attention + MLP with residual adds."""
    y = layers.LayerNormalization(epsilon=1e-6, name=f'{name_prefix}_ln1')(x)
    y = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim,
        dropout=dropout_rate, name=f'{name_prefix}_mha'
    )(y, y)
    x = layers.Add(name=f'{name_prefix}_add1')([y, x])

    y = layers.LayerNormalization(epsilon=1e-6, name=f'{name_prefix}_ln2')(x)
    y = layers.Dense(mlp_dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp1')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop1')(y)
    y = layers.Dense(projection_dim, activation=tf.keras.activations.gelu,
                     name=f'{name_prefix}_mlp2')(y)
    y = layers.Dropout(dropout_rate, name=f'{name_prefix}_drop2')(y)
    return layers.Add(name=f'{name_prefix}_add2')([y, x])


def build_vit_unet_singlehead(
    input_shape, num_classes, inner_patch=3,
    projection_dim=256, num_heads=4, transformer_layers=12,
    mlp_multiplier=2, dropout_rate=0.25, head_units=(512, 256, 128, 64)
):
    """Build a single-head ViT with encoder-decoder skip connections (U-Net style)."""
    num_patches = (input_shape[0] // inner_patch) * (input_shape[1] // inner_patch)

    inputs = keras.Input(shape=input_shape)
    x = PatchExtractor(inner_patch, name='vit_patch_extractor')(inputs)
    x = PatchEncoderWithCLS(num_patches, projection_dim, name='vit_patch_encoder')(x)

    block_list = []
    for i in range(transformer_layers):
        x = transformer_block(
            x, num_heads=num_heads, projection_dim=projection_dim,
            mlp_dim=projection_dim * mlp_multiplier,
            dropout_rate=0.1, name_prefix=f'vit_block_{i+1}'
        )
        if i <= transformer_layers // 2:
            block_list.append(x)
        else:
            x = layers.Add(name=f'vit_skip_add_{i+1}')(
                [x, block_list[transformer_layers - i - 1]]
            )

    x         = layers.Dropout(dropout_rate, name='TRAIN_DROPOUT_1')(x)
    x         = layers.LayerNormalization(epsilon=1e-6, name='vit_cls_norm')(x)
    cls_token = layers.Lambda(lambda t: t[:, 0, :], name='vit_cls_token')(x)

    drop_slots = ['TRAIN_DROPOUT_3', None, 'TRAIN_DROPOUT_5', 'TRAIN_DROPOUT_6']
    head_names = ['vit_head_1', 'vit_head_2', 'vit_head_3', 'vit_head_4']
    y = cls_token
    for units, hname, dname in zip(head_units, head_names, drop_slots):
        y = layers.Dense(units, activation=tf.keras.activations.gelu, name=hname)(y)
        if dname:
            y = layers.Dropout(dropout_rate, name=dname)(y)

    outputs = layers.Dense(num_classes, activation='softmax', name='vit_logits')(y)
    return keras.Model(inputs, outputs, name='ViT_UNet_SingleHead')

# 5.0 — Training Helpers

Utility functions for calibration metrics, optimiser construction, and the formatted Excel report writer.

## 5.1 — Calibration Metrics

In [11]:
def multiclass_brier_score(y_onehot, y_prob):
    """Compute the multiclass Brier score (mean squared probability error)."""
    return float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))


def expected_calibration_error(y_true, y_prob, n_bins=15):
    """Compute Expected Calibration Error (ECE) using equal-width confidence bins."""
    confidences = np.max(y_prob, axis=1)
    predictions = np.argmax(y_prob, axis=1)
    correct     = (predictions == y_true).astype(np.float32)
    bin_edges   = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        in_bin = (confidences >= lo) & (
            confidences <= hi if i == n_bins - 1 else confidences < hi
        )
        prop = np.mean(in_bin)
        if prop > 0:
            ece += np.abs(np.mean(correct[in_bin]) - np.mean(confidences[in_bin])) * prop
    return float(ece)

## 5.5 — Credal Prediction-Set Metrics

Coverage-style metrics for the credal (CREDIT) approach, defined exactly like the conformal-prediction ones: overall empirical coverage, average set size and class-wise empirical coverage. Bounds: `lower = p*`, `upper = min(1, p* + Δp)`.

In [12]:
# ── Credal prediction-set metrics (interval dominance) ─────────────────────────
def credal_set_metrics(lower, upper, y_true, num_classes):
    """
    Turn credal probability bounds into prediction SETS and score them like
    conformal prediction does (overall coverage, average set size, class-wise coverage).

    Set rule (interval dominance): class k stays in C(x) unless another class is
    certainly better, i.e.  k in C(x)  <=>  upper_k(x) >= max_j lower_j(x).
    The argmax-of-lower class is always kept, so no set is ever empty.

    lower, upper : (N, C) credal bounds     y_true : (N,) integer labels 0..C-1
    """
    lower  = np.asarray(lower,  dtype=np.float64)
    upper  = np.maximum(np.asarray(upper, dtype=np.float64), lower)
    y_true = np.asarray(y_true).astype(int)
    n      = len(y_true)

    in_set  = upper >= lower.max(axis=1, keepdims=True)      # (N, C) boolean sets
    covered = in_set[np.arange(n), y_true]                   # true label inside set?

    classwise = np.array([
        covered[y_true == k].mean() if np.any(y_true == k) else np.nan
        for k in range(num_classes)
    ])
    return {
        'n':         int(n),
        'coverage':  float(covered.mean()),                  # overall empirical coverage
        'avg_size':  float(in_set.sum(axis=1).mean()),       # average set size
        'classwise': classwise,                              # class-wise empirical coverage
    }


def credal_frames(records):
    """records: list of (model_name, metrics_dict) -> (overall_df, classwise_df)."""
    overall = pd.DataFrame([{
        'Model':                      m,
        'N_Test_Pixels':              r['n'],
        'Overall_Empirical_Coverage': r['coverage'],
        'Average_Set_Size':           r['avg_size'],
        'Mean_Class_Coverage':        float(np.nanmean(r['classwise'])),
    } for m, r in records])
    classwise = pd.DataFrame([{
        'Model': m,
        **{f'Class_{k + 1}_Coverage': float(v) for k, v in enumerate(r['classwise'])},
    } for m, r in records])
    return overall, classwise


def add_credal_coverage_sheet(wb, overall_df, classwise_df, bounds_note, index=1):
    """Add a 'Credal Coverage' sheet (overall + class-wise tables + method notes)."""
    ws = wb.create_sheet('Credal Coverage', index)
    r = _write_df_to_sheet(ws, overall_df, start_row=1,
                           title='Credal prediction sets — overall empirical coverage & average set size') + 1
    r = _write_df_to_sheet(ws, classwise_df, start_row=r,
                           title='Credal prediction sets — class-wise empirical coverage') + 1
    notes = [
        'How the numbers are computed (same definitions as for conformal prediction):',
        bounds_note,
        'Set rule (interval dominance): class k is in the set  iff  upper_k >= max_j lower_j.',
        'Overall coverage = fraction of labelled test pixels whose true class is inside the set.',
        'Average set size = mean number of classes in the set (1 = confident singleton).',
        'Class-wise coverage (class k) = fraction of test pixels of true class k whose set contains k.',
        'Class_k = reference label k (1-based, same as the reference raster; 0 = unlabelled, excluded).',
        'Coverage here is MEASURED on the test set; unlike (split) conformal prediction it is not calibrated to a target 1-alpha.',
    ]
    for line in notes:
        ws.cell(row=r, column=1, value=line).font = Font(name='Arial', size=10, bold=(line == notes[0]))
        r += 1
    return ws

## 5.2 — Optimiser & LR Schedule

In [13]:
def make_adamw_optimizer(num_train_samples):
    """Build a cosine-decay AdamW optimiser (used for GFNet and ViT)."""
    steps_per_epoch = int(np.ceil(num_train_samples / BATCH_SIZE))
    decay_steps     = max(1, steps_per_epoch * EPOCHS)
    lr_schedule = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LEARNING_RATE,
        decay_steps=decay_steps,
        alpha=TRAIN_CFG['cosine_alpha'],
    )
    return keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=TRAIN_CFG['weight_decay'],
        clipnorm=TRAIN_CFG['clipnorm'],
    )


def _alexnet_legacy_lr(epoch):
    """Cosine LR schedule oscillating between ALEXNET_LR_MIN and ALEXNET_LR_MAX."""
    if EPOCHS <= 1:
        return ALEXNET_LR_START
    phase        = np.pi * epoch / (EPOCHS - 1)
    cosine_decay = 0.5 * (1.0 + np.cos(phase))
    return float((ALEXNET_LR_MAX - ALEXNET_LR_MIN) * cosine_decay + ALEXNET_LR_MIN)

## 5.3 — Excel Report Helpers

Style constants and helper functions for writing DataFrames and plots into a formatted `.xlsx` workbook.

In [14]:
# ── Style constants ────────────────────────────────────────────────────────────
_HDR_FILL   = PatternFill('solid', start_color='1F4E79')
_ALT_FILL   = PatternFill('solid', start_color='D6E4F0')
_HDR_FONT   = Font(name='Arial', bold=True, color='FFFFFF', size=11)
_BODY_FONT  = Font(name='Arial', size=10)
_TITLE_FONT = Font(name='Arial', bold=True, size=13)
_CENTER     = Alignment(horizontal='center', vertical='center', wrap_text=True)
_LEFT       = Alignment(horizontal='left',   vertical='center')
_THIN_SIDE  = Side(style='thin', color='AAAAAA')
_THIN_BORDER = Border(
    left=_THIN_SIDE, right=_THIN_SIDE, top=_THIN_SIDE, bottom=_THIN_SIDE
)


def _style_header_row(ws, row, col_start, col_end):
    """Apply dark-blue header styling to a worksheet row range."""
    for c in range(col_start, col_end + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill      = _HDR_FILL
        cell.font      = _HDR_FONT
        cell.alignment = _CENTER
        cell.border    = _THIN_BORDER


def _style_data_rows(ws, row_start, row_end, col_start, col_end):
    """Apply alternating-row fill and border to a worksheet data range."""
    for r in range(row_start, row_end + 1):
        fill = _ALT_FILL if r % 2 == 0 else PatternFill()
        for c in range(col_start, col_end + 1):
            cell = ws.cell(row=r, column=c)
            cell.font      = _BODY_FONT
            cell.fill      = fill
            cell.alignment = _LEFT
            cell.border    = _THIN_BORDER


def _write_df_to_sheet(ws, df, start_row=1, start_col=1, title=None):
    """Write a DataFrame into a worksheet with headers, alternating rows, and auto-width columns."""
    r = start_row
    if title:
        cell = ws.cell(row=r, column=start_col, value=title)
        cell.font      = _TITLE_FONT
        cell.alignment = _LEFT
        # Guard: merge_cells requires end_column >= start_column (>= 1)
        _end_col = start_col + len(df.columns) - 1
        if _end_col >= start_col:
            ws.merge_cells(
                start_row=r, start_column=start_col,
                end_row=r, end_column=_end_col
            )
        r += 1

    # Header row
    for j, col_name in enumerate(df.columns, start=start_col):
        ws.cell(row=r, column=j, value=col_name)
    _style_header_row(ws, r, start_col, start_col + len(df.columns) - 1)
    r += 1

    # Data rows
    data_start = r
    for _, row_data in df.iterrows():
        for j, val in enumerate(row_data, start=start_col):
            ws.cell(row=r, column=j,
                    value=round(float(val), 6)
                    if isinstance(val, (float, np.floating)) else val)
        r += 1
    _style_data_rows(ws, data_start, r - 1, start_col, start_col + len(df.columns) - 1)

    # Auto column width
    for j, col_name in enumerate(df.columns, start=start_col):
        max_len = max(
            len(str(col_name)),
            max((len(str(ws.cell(row=row, column=j).value or ''))
                 for row in range(data_start, r)), default=0)
        )
        ws.column_dimensions[get_column_letter(j)].width = min(max_len + 4, 40)

    return r  # next free row

## 5.4 — Excel Report Builder

Assembles the full `CREDIT_Results.xlsx` workbook with four sheets: Training Summary, Evaluation Summary, Confusion Matrices, and Plots.

In [15]:
def create_credit_excel_report(
    results_dir,
    train_summary_df,
    eval_summary_df,
    credit_artifacts_dict,
    plot_paths,           # list of (label_str, image_path_str)
    pixel_uncertainty=None,  # dict from prepare_pixel_uncertainty() per model (optional)
    credal_overall_df=None,  # overall coverage / avg set size per model (optional)
    credal_classwise_df=None # class-wise coverage per model (optional)
):
    """
    Create CREDIT_Results.xlsx with these sheets (+ 'Credal Coverage' when provided):
      Sheet 1 — Training Summary
      Sheet 2 — Evaluation Summary
      Sheet 3 — Confusion Matrices (per model, side by side)
      Sheet 4 — Plots (one image per row)
    """
    xlsx_path = Path(results_dir) / 'CREDIT_Results.xlsx'
    wb = Workbook()

    # ── Sheet 1: Training Summary ──────────────────────────────────────────────
    ws1 = wb.active
    ws1.title = 'Training Summary'
    ws1.row_dimensions[1].height = 30
    _write_df_to_sheet(ws1, train_summary_df,
                       start_row=1, title='CREDIT — Training Uncertainty Summary')

    # ── Sheet 2: Evaluation Summary ────────────────────────────────────────────
    ws2 = wb.create_sheet('Evaluation Summary')
    _write_df_to_sheet(ws2, eval_summary_df,
                       start_row=1, title='CREDIT — Full Evaluation Metrics')

    # ── Credal coverage sheet (overall coverage, avg set size, class-wise coverage) ──
    if credal_overall_df is not None and credal_classwise_df is not None:
        add_credal_coverage_sheet(
            wb, credal_overall_df, credal_classwise_df,
            bounds_note='Credal bounds per pixel: lower = p* (student softmax head), upper = min(1, p* + delta_p) (student sigmoid head).',
            index=2)

    # ── Sheet 3: Confusion Matrices ────────────────────────────────────────────
    ws3 = wb.create_sheet('Confusion Matrices')
    col_cursor = 1
    for mname, art in credit_artifacts_dict.items():
        cm    = art['confusion_matrix']
        rep   = art['report']
        n_cls = cm.shape[0]

        # Title
        cell = ws3.cell(row=1, column=col_cursor,
                        value=f'{mname} — Confusion Matrix')
        cell.font = _TITLE_FONT
        ws3.merge_cells(start_row=1, start_column=col_cursor,
                        end_row=1, end_column=col_cursor + n_cls)

        # Column headers (Predicted)
        ws3.cell(row=2, column=col_cursor,
                 value=r'True \ Pred').font = Font(bold=True, name='Arial')
        for j in range(n_cls):
            c = ws3.cell(row=2, column=col_cursor + 1 + j, value=f'C{j+1}')
            c.font      = Font(bold=True, name='Arial', color='FFFFFF')
            c.fill      = _HDR_FILL
            c.alignment = _CENTER

        # Data rows
        for i in range(n_cls):
            ws3.cell(row=3 + i, column=col_cursor,
                     value=f'C{i+1}').font = Font(bold=True, name='Arial')
            for j in range(n_cls):
                val  = int(cm[i, j])
                cell = ws3.cell(row=3 + i, column=col_cursor + 1 + j, value=val)
                cell.alignment = _CENTER
                cell.border    = _THIN_BORDER
                if i == j:  # diagonal → green
                    cell.fill = PatternFill('solid', start_color='C6EFCE')
                    cell.font = Font(name='Arial', bold=True, color='276221')
                else:
                    cell.font = _BODY_FONT

        # Per-class F1 from classification report
        f1_row = 3 + n_cls + 1
        ws3.cell(row=f1_row, column=col_cursor,
                 value='Per-Class F1').font = Font(bold=True, name='Arial')
        for j in range(n_cls):
            key = str(j)
            f1  = rep.get(key, {}).get('f1-score', '')
            ws3.cell(row=f1_row, column=col_cursor + 1 + j,
                     value=round(f1, 4) if f1 != '' else '').font = _BODY_FONT

        col_cursor += n_cls + 3   # gap between model tables

    # ── Sheet 4: Plots ─────────────────────────────────────────────────────────
    ws4 = wb.create_sheet('Plots')
    title_cell = ws4.cell(row=1, column=1, value='CREDIT — All Output Plots')
    title_cell.font = Font(name='Arial', bold=True, size=14)

    img_row = 3
    for label, img_path in plot_paths:
        if not Path(img_path).exists():
            continue
        ws4.cell(row=img_row, column=1,
                 value=label).font = Font(name='Arial', bold=True, size=11)
        img_row += 1

        xl_img = XLImage(img_path)
        orig_w, orig_h = xl_img.width, xl_img.height
        target_w = 900
        scale         = target_w / orig_w if orig_w > 0 else 1
        xl_img.width  = int(orig_w * scale)
        xl_img.height = int(orig_h * scale)

        cell_addr = f'A{img_row}'
        ws4.add_image(xl_img, cell_addr)
        # openpyxl row height is in points; 1pt ≈ 0.75px
        ws4.row_dimensions[img_row].height = xl_img.height * 0.75
        img_row += int(xl_img.height / 15) + 3

        # ── Extra sheets: per-pixel AU / EU / TU + threshold analysis ──────────────
    if pixel_uncertainty:
        add_pixel_uncertainty_sheets(wb, pixel_uncertainty)

    wb.save(xlsx_path)
    print(f'\n✅ Excel report saved → {xlsx_path}')
    return str(xlsx_path)

save_credit_excel_report = create_credit_excel_report


# 6.0 — CREDIT Distillation

Generates soft targets from the pre-trained ensemble teachers, builds a dual-head student for each architecture, and trains it via KL divergence (aleatoric head) + MSE (epistemic head).

## 6.1 — Soft Target Generation

Loads each ensemble teacher, runs inference over the training data, and derives CREDIT targets: `p_star` (normalised per-class minimum — aleatoric proxy) and `delta_p` (per-class prediction spread — epistemic proxy).

In [16]:
CUSTOM_OBJECTS = {
    "PatchExtractor":       PatchExtractor,
    "PatchPositionEncoder": PatchPositionEncoder,
    "GlobalFilterLayer":    GlobalFilterLayer,
    "PatchEncoderWithCLS":  PatchEncoderWithCLS,
}


def _dedup_by_filename(paths):
    """Keep one path per unique filename (shallowest path first), sorted by filename."""
    seen = set()
    unique = []
    for p in sorted(paths, key=lambda x: (len(Path(x).parts), str(x))):
        name = Path(p).name
        if name not in seen:
            seen.add(name)
            unique.append(str(p))
    return sorted(unique, key=lambda x: Path(x).name)


def get_ensemble_paths(model_name):
    """Locate the 5 ensemble teacher .keras files for a given architecture."""
    _OBF_MAP = {
        "AlexNet_CNN": ["k016", "k017", "k018", "k019", "k020"],
        "GFNet":       ["k021", "k022", "k023", "k024", "k025"],
        "ViT_UNet":    ["k026", "k027", "k028", "k029", "k030"],
    }

    search_patterns = [
        f"/kaggle/input/cuq6-ensemble-k04/**/{model_name}_ens_*_final.keras",
        f"/kaggle/input/notebooks/daneshselwal/cuq6-ensemble-k04/**/{model_name}_ens_*_final.keras",
        f"/kaggle/input/**/{model_name}_ens_*_final.keras",
    ]
    if ENSEMBLE_DIR is not None:
        search_patterns.insert(0, str(ENSEMBLE_DIR / f"{model_name}_ens_*_final.keras"))
        search_patterns.insert(1, str(ENSEMBLE_DIR / "**" / f"{model_name}_ens_*_final.keras"))

    for pattern in search_patterns:
        found = glob.glob(pattern, recursive=True)
        if found:
            return _dedup_by_filename(found)

    # Fallback: obfuscated names across ENSEMBLE_DIR and /kaggle/input
    if model_name in _OBF_MAP:
        obf_found = []
        for code in _OBF_MAP[model_name]:
            obf_found.extend(glob.glob(f"/kaggle/input/**/{code}.keras", recursive=True))
        if obf_found:
            return _dedup_by_filename(obf_found)

    # Local project fallback
    local_candidates = [
        Path("../ensemble/models"),
        Path("Classification/ensemble/models"),
        Path("./models"),
        Path("Classification/ensemble/models/ensembles"),
    ]
    for lc in local_candidates:
        if lc.exists():
            found = glob.glob(str(lc / "**" / f"{model_name}_ens_*_final.keras"), recursive=True)
            if found:
                return _dedup_by_filename(found)

    return []


def generate_credit_targets(ensemble_paths, x_data, batch_size=128):
    """
    Run each teacher over x_data and derive CREDIT interval targets.

    Returns:
        p_star_true  : (N, C) normalised per-class minimum  (aleatoric proxy)
        delta_p_true : (N, C) per-class prediction spread   (epistemic proxy)
    """
    import gc
    all_preds = []
    for path in ensemble_paths:
        print(f"  Loading teacher: {path}")
        model = tf.keras.models.load_model(
            path, compile=False, custom_objects=CUSTOM_OBJECTS, safe_mode=False
        )
        all_preds.append(model.predict(x_data, batch_size=batch_size, verbose=0))
        del model
        tf.keras.backend.clear_session()
        gc.collect()

    stacked = tf.stack(all_preds, axis=0)   # (M, N, C)
    p_min   = tf.reduce_min(stacked, axis=0)  # (N, C)
    p_max   = tf.reduce_max(stacked, axis=0)  # (N, C)

    delta_p_true = p_max - p_min
    p_star_true  = p_min / (tf.reduce_sum(p_min, axis=-1, keepdims=True) + 1e-12)
    return p_star_true, delta_p_true


## 6.2 — CREDIT Student (Dual-Head)

Wraps any single-head base architecture with two output heads: a softmax head for the aleatoric belief (`p_star`) and a sigmoid head for the epistemic spread (`delta_p`).

In [17]:
def build_credit_student(base_builder_func, num_classes):
    """
    Attach CREDIT dual-output heads to any single-head base architecture.

    Args:
        base_builder_func: Zero-argument callable that returns a Keras model.
        num_classes:       Number of target classes.

    Returns:
        A Keras model with outputs [p_star (softmax), delta_p (sigmoid)].
    """
    base_model = base_builder_func()
    features   = base_model.layers[-2].output   # penultimate feature tensor

    p_star  = layers.Dense(num_classes, activation='softmax',  name='p_star' )(features)
    delta_p = layers.Dense(num_classes, activation='sigmoid',  name='delta_p')(features)

    return tf.keras.Model(
        inputs=base_model.input,
        outputs=[p_star, delta_p],
        name='CREDIT_Student'
    )

## 6.3 — Model Builder Registry

Centralised registry mapping model names to zero-argument builder lambdas.

In [18]:
model_builders = {
    'AlexNet_CNN': lambda: build_alexnet(
        input_shape, num_classes,
        dropout_rate=DROPOUT_RATE, cfg=ALEXNET_CFG
    ),
    'GFNet': lambda: build_gfnet(
        input_shape, num_classes,
        inner_patch=INNER_PATCH,
        hidden_dim=GFNET_CFG['hidden_dim'],
        num_blocks=GFNET_CFG['num_blocks'],
        mlp_ratio=GFNET_CFG['mlp_ratio'],
        dropout_rate=DROPOUT_RATE
    ),
    'ViT_UNet': lambda: build_vit_unet_singlehead(
        input_shape, num_classes,
        inner_patch=INNER_PATCH,
        projection_dim=VIT_CFG['projection_dim'],
        num_heads=VIT_CFG['num_heads'],
        transformer_layers=VIT_CFG['transformer_layers'],
        mlp_multiplier=VIT_CFG['mlp_multiplier'],
        dropout_rate=DROPOUT_RATE,
        head_units=tuple(VIT_CFG['head_units'])
    ),
}

## 6.4 — Verify Ensemble Teachers Available

In [19]:
# Debug: list what's available in kaggle input
import subprocess
result = subprocess.run(['find', '/kaggle/input', '-maxdepth', '4', '-name', '*.keras', '-o', '-name', '*.csv'], capture_output=True, text=True)
print('Input files found:')
print(result.stdout[:2000])
print('---')

# Verify all ensemble teacher models are accessible before training
print('=== Verifying ensemble teacher model availability ===')
all_found = True
for mname in model_builders:
    paths = get_ensemble_paths(mname)
    status = '✅' if len(paths) == 5 else ('⚠️' if paths else '❌')
    print(f'  {status} {mname}: {len(paths)} models found')
    for p in paths:
        sz = Path(p).stat().st_size / 1024 / 1024
        print(f'      {Path(p).name} ({sz:.1f} MB)')
    if len(paths) == 0:
        all_found = False

if not all_found:
    print('\n⚠️  Some ensemble models missing – listing all .keras files in input:')
    import subprocess
    result = subprocess.run(['find', '/kaggle/input', '-name', '*.keras'],
                           capture_output=True, text=True)
    print(result.stdout[:3000])


Input files found:
/kaggle/input/dias-6-bands-data/data.csv
/kaggle/input/dias-6-bands-data/k001.csv
/kaggle/input/dias-6-bands-data/k002.csv
/kaggle/input/dias-6-bands-data/ref.csv
/kaggle/input/cuq6-ensemble-k04/ensemble/models/classification_summary.csv

---
=== Verifying ensemble teacher model availability ===
  ✅ AlexNet_CNN: 5 models found
      AlexNet_CNN_ens_1_final.keras (259.5 MB)
      AlexNet_CNN_ens_2_final.keras (259.5 MB)
      AlexNet_CNN_ens_3_final.keras (259.5 MB)
      AlexNet_CNN_ens_4_final.keras (259.5 MB)
      AlexNet_CNN_ens_5_final.keras (259.5 MB)
  ✅ GFNet: 5 models found
      GFNet_ens_1_final.keras (121.4 MB)
      GFNet_ens_2_final.keras (121.4 MB)
      GFNet_ens_3_final.keras (121.4 MB)
      GFNet_ens_4_final.keras (121.4 MB)
      GFNet_ens_5_final.keras (121.4 MB)
  ✅ ViT_UNet: 5 models found
      ViT_UNet_ens_1_final.keras (185.1 MB)
      ViT_UNet_ens_2_final.keras (185.1 MB)
      ViT_UNet_ens_3_final.keras (185.1 MB)
      ViT_UNet_ens_4_fina

## 6.5 — Training Loop

For each architecture: locate ensemble teachers → generate soft targets → build `tf.data` pipelines → build and compile the CREDIT student → train with `ModelCheckpoint` → compute a quick post-training uncertainty summary.

In [20]:
train_results = []

for model_name, builder in model_builders.items():
    print(f"\n{'='*25} CREDIT Distillation: {model_name} {'='*25}")

    # ── 1. Locate ensemble teachers ────────────────────────────────────────────
    ensemble_paths = get_ensemble_paths(model_name)
    if len(ensemble_paths) == 0:
        print('  No ensemble models found — skipping.')
        continue
    if len(ensemble_paths) < 5:
        print(f'  Warning: only {len(ensemble_paths)} ensemble models found (expected 5).')

    # ── 2. Select correct data split ───────────────────────────────────────────
    x_tr, x_te = (
        (x_train_alex, x_test_alex) if model_name == 'AlexNet_CNN'
        else (x_train, x_test)
    )

    # ── 3. Generate soft targets ───────────────────────────────────────────────
    print('  Generating training targets...')
    p_star_tr, delta_p_tr = generate_credit_targets(ensemble_paths, x_tr)

    print('  Generating test targets...')
    p_star_te, delta_p_te = generate_credit_targets(ensemble_paths, x_te)

    # ── 4. Build tf.data pipelines ─────────────────────────────────────────────
    train_ds = (
        tf.data.Dataset
        .from_tensor_slices((x_tr, (p_star_tr, delta_p_tr)))
        .shuffle(1024).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    )
    test_ds = (
        tf.data.Dataset
        .from_tensor_slices((x_te, (p_star_te, delta_p_te)))
        .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    )

    # ── 5. Build & compile student ─────────────────────────────────────────────
    tf.keras.backend.clear_session()
    student = build_credit_student(builder, num_classes)

    optimizer = (
        keras.optimizers.Adagrad(learning_rate=ALEXNET_LR_START)
        if model_name == 'AlexNet_CNN'
        else keras.optimizers.Adam(learning_rate=LEARNING_RATE)
    )

    student.compile(
        optimizer=optimizer,
        loss={
            'p_star':  tf.keras.losses.KLDivergence(),
            'delta_p': tf.keras.losses.MeanSquaredError(),
        },
        loss_weights={'p_star': 1.0, 'delta_p': 0.5},   # lambda = 0.5
    )

    # ── 6. Train ───────────────────────────────────────────────────────────────
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'
    callbacks = [
        keras.callbacks.ModelCheckpoint(
            filepath=str(best_path), monitor='val_loss',
            mode='min', save_best_only=True, verbose=0
        )
    ]

    t0 = time.perf_counter()
    student.fit(
        train_ds, validation_data=test_ds,
        epochs=EPOCHS, callbacks=callbacks, verbose=2
    )
    train_time = time.perf_counter() - t0

    # ── 7. Post-training uncertainty summary ───────────────────────────────────
    student.load_weights(best_path)
    p_star_pred, delta_p_pred = student.predict(x_te, batch_size=BATCH_SIZE, verbose=0)

    au = -np.sum(p_star_pred * np.log(p_star_pred + 1e-12), axis=-1)
    eu =  np.mean(delta_p_pred, axis=-1)
    tu =  au + eu

    train_results.append({
        'Model':          model_name,
        'Mean_AU':        np.mean(au),
        'Mean_EU':        np.mean(eu),
        'Mean_TU':        np.mean(tu),
        'Train_Time_sec': train_time,
    })

# ── Save training uncertainty summary ─────────────────────────────────────────
summary_df   = pd.DataFrame(train_results)
summary_path = RESULTS_DIR / 'credit_train_uncertainty_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f'\nSaved training summary → {summary_path}')
print(summary_df)

# Stash for the final Excel report
_credit_train_summary_df = summary_df.copy()


========================= CREDIT Distillation: AlexNet_CNN =========================
  Generating training targets...
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_1_final.keras


I0000 00:00:1793766138.774676      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1793766138.778448      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1793766146.892680      72 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_2_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_3_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_4_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_5_final.keras
  Generating test targets...
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_1_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_2_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_3_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_4_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/AlexNet_CNN_ens_5_final.keras
Epoch 1/

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'gf_patch_encoder', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_2_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_3_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_4_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_5_final.keras
  Generating test targets...
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_1_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_2_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_3_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_4_final.keras
  Loading teacher: /kaggle/input/cuq6-ensemble-k04/ensemble/models/ensembles/GFNet_ens_5_final.keras
Epoch 1/100
81/81 - 45s - 562ms/step - delta_p_loss: 0.0166 - 

# 7.0 — Evaluation

Loads the best saved CREDIT student weights for each architecture and computes a full suite of classification and calibration metrics — no re-training.

## 7.1 — Load, Predict & Compute Metrics

In [21]:
eval_results     = []
credit_artifacts = {}
credal_records   = []   # (model_name, credal_set_metrics dict) for the coverage sheet

for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'

    if not best_path.exists():
        print(f'  [{model_name}] No saved weights at {best_path} — skipping.')
        continue

    print(f"\n{'='*25} Evaluating: {model_name} {'='*25}")
    tf.keras.backend.clear_session()

    student = build_credit_student(builder, num_classes)
    student.load_weights(best_path)

    # Select the correct test split for this architecture
    if model_name == 'AlexNet_CNN':
        x_te, y_te_true, y_te_cat = x_test_alex, y_test_alex, y_test_alex_cat
    else:
        x_te, y_te_true, y_te_cat = x_test, y_test, y_test_cat

    p_star_pred, delta_p_pred = student.predict(x_te, batch_size=BATCH_SIZE, verbose=0)

    au = -np.sum(p_star_pred * np.log(p_star_pred + 1e-12), axis=-1)
    eu =  np.mean(delta_p_pred, axis=-1)
    tu =  au + eu

    y_pred = np.argmax(p_star_pred, axis=-1)

    # Credal prediction sets: lower = p*, upper = min(1, p* + delta_p)
    credal_m = credal_set_metrics(
        lower=p_star_pred,
        upper=np.minimum(1.0, p_star_pred + delta_p_pred),
        y_true=y_te_true, num_classes=num_classes
    )
    credal_records.append((model_name, credal_m))

    credit_artifacts[model_name] = {
        'confusion_matrix': confusion_matrix(y_te_true, y_pred),
        'report':           classification_report(
            y_te_true, y_pred, output_dict=True, zero_division=0
        ),
    }

    eval_results.append({
        'Model':         model_name,
        'Test_Accuracy': float(accuracy_score(y_te_true, y_pred)),
        'Macro_F1':      float(f1_score(y_te_true, y_pred, average='macro')),
        'Cohen_Kappa':   float(cohen_kappa_score(y_te_true, y_pred)),
        'Test_NLL':      float(log_loss(y_te_true, p_star_pred,
                                        labels=np.arange(num_classes))),
        'Test_Brier':    float(multiclass_brier_score(y_te_cat, p_star_pred)),
        'Test_ECE':      float(expected_calibration_error(
            y_te_true, p_star_pred, n_bins=15)),
        'Mean_AU':       float(np.mean(au)),
        'Mean_EU':       float(np.mean(eu)),
        'Mean_TU':       float(np.mean(tu)),
        'Credal_Coverage':     credal_m['coverage'],
        'Credal_Avg_Set_Size': credal_m['avg_size'],
    })

eval_df = pd.DataFrame(eval_results)
eval_df.to_csv(RESULTS_DIR / 'credit_evaluation_summary.csv', index=False)
print('\n--- CREDIT Evaluation Summary ---')
print(eval_df.to_string(index=False))

# Credal coverage tables (overall + class-wise)
_credal_overall_df, _credal_classwise_df = credal_frames(credal_records)
_credal_overall_df.to_csv(RESULTS_DIR / 'credit_credal_coverage_overall.csv', index=False)
_credal_classwise_df.to_csv(RESULTS_DIR / 'credit_credal_coverage_classwise.csv', index=False)
print('\n--- CREDIT Credal Coverage (overall) ---')
print(_credal_overall_df.to_string(index=False))
print('\n--- CREDIT Credal Coverage (class-wise) ---')
print(_credal_classwise_df.to_string(index=False))

# Stash for the final Excel report
_credit_eval_df        = eval_df.copy()
_credit_artifacts_dict = credit_artifacts


========================= Evaluating: AlexNet_CNN =========================

========================= Evaluating: GFNet =========================

========================= Evaluating: ViT_UNet =========================

--- CREDIT Evaluation Summary ---
      Model  Test_Accuracy  Macro_F1  Cohen_Kappa  Test_NLL  Test_Brier  Test_ECE  Mean_AU  Mean_EU  Mean_TU  Credal_Coverage  Credal_Avg_Set_Size
AlexNet_CNN       0.995592  0.995011     0.994022  0.013644    0.007309  0.002620 0.019521 0.001219 0.020740         0.995592                  1.0
      GFNet       0.997448  0.996092     0.996539  0.012935    0.004670  0.001325 0.014228 0.000070 0.014298         0.997448                  1.0
   ViT_UNet       0.989095  0.982603     0.985211  0.067603    0.018795  0.034172 0.228748 0.004286 0.233034         0.989095                  1.0

--- CREDIT Credal Coverage (overall) ---
      Model  N_Test_Pixels  Overall_Empirical_Coverage  Average_Set_Size  Mean_Class_Coverage
AlexNet_CNN        

## 7.2 — Confusion Matrices

Renders and saves a side-by-side confusion matrix heatmap for every evaluated model.

In [22]:
if credit_artifacts:
    class_ticks = [str(i + 1) for i in range(num_classes)]
    n_models    = len(credit_artifacts)
    fig, axes   = plt.subplots(1, n_models, figsize=(7 * n_models, 5.5))
    if n_models == 1:
        axes = [axes]

    for ax, (mname, art) in zip(axes, credit_artifacts.items()):
        sns.heatmap(
            art['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=class_ticks, yticklabels=class_ticks, cbar=False, ax=ax
        )
        ax.set_title(f'{mname} — CREDIT\nConfusion Matrix', fontsize=13)
        ax.set_xlabel('Predicted Class')
        ax.set_ylabel('True Class')

    plt.tight_layout()
    save_path = RESULTS_DIR / 'credit_confusion_matrices.png'
    plt.savefig(save_path, dpi=220, bbox_inches='tight')
    print(f'Saved → {save_path}')
    plt.close()

Saved → /kaggle/working/credit/results/credit_confusion_matrices.png


# 8.0 — Spatial Uncertainty Mapping

Runs full-scene inference for each CREDIT student and produces a 3×4 figure showing: the base prediction map, binary certain/uncertain masks, grey-overlay maps, and per-class pixel-count bar charts — all for the aleatoric, epistemic, and total uncertainty channels.

## 8.1 — Extract Full-Scene Patch Array

In [23]:
print('Extracting full-scene patch array (all H×W pixels)...')
pad   = PATCH_SIZE // 2
x_pad = np.pad(x_img, ((pad, pad), (pad, pad), (0, 0)), mode='edge')

scene_patches = np.empty((H * W, PATCH_SIZE, PATCH_SIZE, B), dtype=np.float32)
for idx, (r, c) in enumerate(
    [(r, c) for r in range(H) for c in range(W)]
):
    scene_patches[idx] = x_pad[r:r + PATCH_SIZE, c:c + PATCH_SIZE, :]

print(f'Scene patch tensor: {scene_patches.shape}')

Extracting full-scene patch array (all H×W pixels)...
Scene patch tensor: (101310, 9, 9, 6)


## 8.2 — Spatial Map Generator

Defines the function that produces and saves the 3×4 uncertainty figure for a given student model.

In [24]:
import colorsys

CLASS_COLOR_BASE = [
    '#0000FF', '#00FF00', '#FF0000', '#00FFFF',
    '#FF00FF', '#FFFF00', '#A52A2A', '#FFA500',
    '#7FFF00', '#8A2BE2',
]
UNCERTAIN_COLOR = '#808080'

def _generate_extra_shades(n_extra, hue_offset=0.05, sat=0.85, val=0.95):
    """Bold, highly-saturated extra hues in the same style as CLASS_COLOR_BASE."""
    shades = []
    for i in range(n_extra):
        hue = (hue_offset + i / max(n_extra, 1)) % 1.0
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        shades.append('#%02x%02x%02x' % (int(r * 255), int(g * 255), int(b * 255)))
    return shades

def get_class_colors(n):
    """Return n distinct hex colours, extending the base 10 with additional bold
    shades in the same saturated style if needed."""
    if n <= len(CLASS_COLOR_BASE):
        return CLASS_COLOR_BASE[:n]
    return CLASS_COLOR_BASE + _generate_extra_shades(n - len(CLASS_COLOR_BASE))

def resolve_uncertain_color(class_colors, preferred=UNCERTAIN_COLOR, min_dist=60):
    """Return `preferred` unless it visually clashes with any class colour, in
    which case fall back to a distinct dark slate so 'Uncertain' never collides
    with a real class colour."""
    def _hex_to_rgb(h):
        h = h.lstrip('#')
        return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    pr, pg, pb = _hex_to_rgb(preferred)
    for c in class_colors:
        cr, cg, cb = _hex_to_rgb(c)
        dist = ((pr - cr) ** 2 + (pg - cg) ** 2 + (pb - cb) ** 2) ** 0.5
        if dist < min_dist:
            return '#2F2F2F'
    return preferred


def generate_spatial_credit_maps(
    model_name, student_model, scene_pixels,
    H, W, au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7
):
    """
    Run full-scene inference and save a 3×4 spatial uncertainty figure.

    Panel layout (rows × cols):
      Row 0: Base prediction | AU binary | EU binary | TU binary
      Row 1: AU grey overlay | EU grey overlay | TU grey overlay | (blank)
      Row 2: AU bar chart   | EU bar chart   | TU bar chart   | (blank)
    """
    print(f'  Inferring full scene for {model_name}...')
    p_star_scene, delta_p_scene = student_model.predict(
        scene_pixels, batch_size=2048, verbose=0
    )
    p_star_scene = np.clip(p_star_scene, 1e-12, 1.0)
    n_cls = p_star_scene.shape[-1]

    au_scene = -np.sum(p_star_scene * np.log(p_star_scene), axis=-1)
    eu_scene =  np.mean(delta_p_scene, axis=-1)
    tu_scene =  au_scene + eu_scene

    pred_map = np.argmax(p_star_scene, axis=-1).reshape(H, W)
    au_mask  = (au_scene.reshape(H, W) > au_thresh).astype(int)
    eu_mask  = (eu_scene.reshape(H, W) > eu_thresh).astype(int)
    tu_mask  = (tu_scene.reshape(H, W) > tu_thresh).astype(int)

    combined_au = np.where(au_mask == 1, n_cls, pred_map)
    combined_eu = np.where(eu_mask == 1, n_cls, pred_map)
    combined_tu = np.where(tu_mask == 1, n_cls, pred_map)

    class_colors = get_class_colors(n_cls)
    uncertain_color = resolve_uncertain_color(class_colors)
    cmap_base   = ListedColormap(class_colors)
    cmap_unc    = ListedColormap(class_colors + [uncertain_color])
    cmap_binary = ListedColormap(['#FFFF00', '#001F3F'])

    bar_lbls = [f'Class {i}' for i in range(n_cls)] + ['Uncertain']
    bar_cols = class_colors + [uncertain_color]

    fig, axes = plt.subplots(3, 4, figsize=(38, 26))
    fig.suptitle(
        f'{model_name} — CREDIT Uncertainty Maps (Absolute Thresholds)',
        fontsize=24, fontweight='bold', y=0.99
    )

    # ── Row 0: Base prediction + 3 binary maps ─────────────────────────────────
    axes[0, 0].imshow(pred_map, cmap=cmap_base, vmin=0, vmax=n_cls - 1)
    axes[0, 0].set_title('Base Prediction Map', fontsize=15)
    axes[0, 0].axis('off')

    binary_specs = [
        (axes[0, 1], au_mask, f'Aleatoric (AU > {au_thresh})'),
        (axes[0, 2], eu_mask, f'Epistemic (EU > {eu_thresh})'),
        (axes[0, 3], tu_mask, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, mask, label in binary_specs:
        ax.imshow(mask, cmap=cmap_binary, vmin=0, vmax=1)
        ax.set_title(f'Certain vs Uncertain\n{label}', fontsize=15, pad=10)
        ax.axis('off')
        ax.legend(
            handles=[
                Patch(facecolor='#FFFF00', label='Certain'),
                Patch(facecolor='#001F3F', label='Uncertain'),
            ],
            loc='upper left',
            bbox_to_anchor=(0.0, -0.02),
            borderaxespad=0,
            fontsize=11,
            framealpha=0.9,
            ncol=2
        )

    # ── Row 1: Grey overlay maps ───────────────────────────────────────────────
    overlay_specs = [
        (axes[1, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[1, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[1, 2], combined_tu, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, combined, label in overlay_specs:
        ax.imshow(combined, cmap=cmap_unc, vmin=0, vmax=n_cls)
        ax.set_title(f'Grey Overlay — {label}', fontsize=15, pad=10)
        ax.axis('off')
    axes[1, 3].axis('off')

    # ── Row 2: Pixel-count bar charts ──────────────────────────────────────────
    bar_specs = [
        (axes[2, 0], combined_au, f'Aleatoric (AU > {au_thresh})'),
        (axes[2, 1], combined_eu, f'Epistemic (EU > {eu_thresh})'),
        (axes[2, 2], combined_tu, f'Total     (TU > {tu_thresh})'),
    ]
    for ax, combined, label in bar_specs:
        uniq, cnt = np.unique(combined, return_counts=True)
        c_dict    = {int(k): int(v) for k, v in zip(uniq, cnt)}
        bar_vals  = [c_dict.get(i, 0) for i in range(n_cls + 1)]
        ax.bar(bar_lbls, bar_vals, color=bar_cols, edgecolor='black')
        ax.set_title(f'Pixel Counts — {label}', fontsize=15, pad=10)
        ax.tick_params(axis='x', rotation=45, labelsize=11)
        ax.set_ylabel('Pixel Count', fontsize=12)
        max_val = max(bar_vals, default=1)
        for i, v in enumerate(bar_vals):
            ax.text(i, v + max_val * 0.01,
                    f'{v:,}', ha='center', va='bottom',
                    fontweight='bold', fontsize=9)
        ax.set_ylim(0, max_val * 1.12)
    axes[2, 3].axis('off')

    plt.tight_layout(rect=[0, 0.03, 1, 0.98])
    save_path = RESULTS_DIR / f'{model_name}_CREDIT_spatial_maps.png'
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f'  Saved → {save_path}')
    plt.close()
    
    # Return per-pixel values so they can be exported to Excel (section 9.0)
    return {
        'au': au_scene, 'eu': eu_scene, 'tu': tu_scene,
        'pred': pred_map.reshape(-1),          # 0-based class index per pixel (row-major)
        'H': H, 'W': W, 'n_classes': n_cls,
        'thresholds': (au_thresh, eu_thresh, tu_thresh),
    }

In [25]:
# ── Pixel-level uncertainty table + threshold analysis (for the Excel report) ──
EXCEL_MAX_DATA_ROWS    = 1_048_575            # Excel limit (1,048,576 rows) minus the header row
SENSITIVITY_THRESHOLDS = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]


def build_pixel_uncertainty_df(unc, y_ref):
    """
    One row per pixel that is uncertain under AU, EU or TU (union of the three masks).
    Row / Col / Pixel_ID are 1-based (Pixel_ID = (Row-1)*W + Col, row-major).
    Predicted_Class is 1-based. Reference_Class is the .mat label (0 = unlabelled).
    Sorted by TU, highest first.
    """
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    W = unc['W']
    au_t, eu_t, tu_t = unc['thresholds']

    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t
    idx = np.flatnonzero(f_au | f_eu | f_tu)

    r0, c0 = np.divmod(idx, W)
    tags = np.char.add(
        np.char.add(np.where(f_au[idx], 'AU ', ''), np.where(f_eu[idx], 'EU ', '')),
        np.where(f_tu[idx], 'TU', '')
    )
    under = pd.Series(tags).str.strip().str.replace(' ', '+', regex=False)

    df = pd.DataFrame({
        'Pixel_ID':        idx + 1,
        'Row':             r0 + 1,
        'Col':             c0 + 1,
        'Predicted_Class': unc['pred'][idx] + 1,
        'Reference_Class': np.asarray(y_ref).reshape(-1)[idx],
        'AU':              au[idx].astype(np.float64).round(6),
        'EU':              eu[idx].astype(np.float64).round(6),
        'TU':              tu[idx].astype(np.float64).round(6),
        f'AU_Uncertain (>{au_t})': f_au[idx].astype(int),
        f'EU_Uncertain (>{eu_t})': f_eu[idx].astype(int),
        f'TU_Uncertain (>{tu_t})': f_tu[idx].astype(int),
        'Uncertain_Under': under.values,
    })
    return df.sort_values('TU', ascending=False, kind='mergesort').reset_index(drop=True)


def build_threshold_analysis(unc):
    """Summary tables: is AU+EU=TU, and how much do the thresholds matter?"""
    au, eu, tu = unc['au'], unc['eu'], unc['tu']
    n = au.size
    au_t, eu_t, tu_t = unc['thresholds']
    f_au, f_eu, f_tu = au > au_t, eu > eu_t, tu > tu_t

    counts = pd.DataFrame({
        'Criterion':        ['AU', 'EU', 'TU'],
        'Threshold':        [au_t, eu_t, tu_t],
        'Uncertain_Pixels': [int(f_au.sum()), int(f_eu.sum()), int(f_tu.sum())],
    })
    counts['Percent_of_Scene'] = (100.0 * counts['Uncertain_Pixels'] / n).round(3)

    regions = [
        ('AU only', 1, 0, 0), ('EU only', 0, 1, 0), ('TU only', 0, 0, 1),
        ('AU & EU only', 1, 1, 0), ('AU & TU only', 1, 0, 1), ('EU & TU only', 0, 1, 1),
        ('AU & EU & TU', 1, 1, 1), ('None (certain under all three)', 0, 0, 0),
    ]
    overlap = pd.DataFrame([
        {'Region (mutually exclusive)': name,
         'Pixels': int(((f_au == bool(a)) & (f_eu == bool(e)) & (f_tu == bool(t))).sum())}
        for name, a, e, t in regions
    ])

    au_vs_tu = pd.DataFrame({
        'Comparison': ['AU-uncertain but NOT TU-uncertain', 'TU-uncertain but NOT AU-uncertain'],
        'Pixels':     [int((f_au & ~f_tu).sum()), int((f_tu & ~f_au).sum())],
    })

    diff = np.abs(au.astype(np.float64) + eu.astype(np.float64) - tu.astype(np.float64))
    identity = pd.DataFrame({
        'Metric': ['max |AU + EU - TU|', 'mean |AU + EU - TU|', 'min (TU - AU)  (= min EU, must be >= 0)',
                   'max possible AU = ln(#classes)', 'max observed AU', 'max observed EU', 'max observed TU'],
        'Value':  [float(diff.max()), float(diff.mean()), float((tu - au).min()),
                   float(np.log(unc['n_classes'])), float(au.max()), float(eu.max()), float(tu.max())],
    })

    sens = pd.DataFrame({'Threshold': SENSITIVITY_THRESHOLDS})
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (pixels)'] = [int((arr > t).sum()) for t in SENSITIVITY_THRESHOLDS]
    for name, arr in (('AU', au), ('EU', eu), ('TU', tu)):
        sens[f'{name} > t (%)'] = [round(100.0 * float((arr > t).mean()), 3) for t in SENSITIVITY_THRESHOLDS]

    return {'counts': counts, 'overlap': overlap, 'au_vs_tu': au_vs_tu,
            'identity': identity, 'sensitivity': sens, 'n_scene_pixels': int(n)}


def prepare_pixel_uncertainty(model_name, unc, y_ref, results_dir):
    """Build the uncertain-pixel table (+ full CSV copy) and threshold analysis for one model."""
    df_full  = build_pixel_uncertainty_df(unc, y_ref)
    csv_path = Path(results_dir) / f'{model_name}_uncertain_pixels.csv'
    df_full.to_csv(csv_path, index=False)

    truncated = len(df_full) > EXCEL_MAX_DATA_ROWS
    df_xl     = df_full.iloc[:EXCEL_MAX_DATA_ROWS] if truncated else df_full
    print(f'  [{model_name}] uncertain pixels (AU|EU|TU): {len(df_full):,}  → full table: {csv_path}')
    if truncated:
        print(f'  ⚠️ [{model_name}] exceeds Excel row limit; sheet holds the top {len(df_xl):,} by TU. '
              f'Use the CSV for the rest.')
    return {
        'df': df_xl, 'n_uncertain_total': len(df_full), 'truncated': truncated,
        'csv_path': str(csv_path), 'analysis': build_threshold_analysis(unc),
        'thresholds': unc['thresholds'],
    }


def add_pixel_uncertainty_sheets(wb, pixel_uncertainty):
    """Add one 'Uncertain Pixels - <model>' sheet per model + one 'Threshold Analysis' sheet."""
    if not pixel_uncertainty:
        return
    for mname, res in pixel_uncertainty.items():
        ws = wb.create_sheet(f'Uncertain Pixels - {mname}'[:31])
        df = res['df']
        ws.append(list(df.columns))
        _style_header_row(ws, 1, 1, len(df.columns))
        ws.row_dimensions[1].height = 32
        for row in df.itertuples(index=False, name=None):
            ws.append(row)
        for j, col in enumerate(df.columns, start=1):
            ws.column_dimensions[get_column_letter(j)].width = max(12, min(len(str(col)) + 4, 26))
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = f'A1:{get_column_letter(len(df.columns))}{max(len(df) + 1, 2)}'

    wa = wb.create_sheet('Threshold Analysis')
    wa.cell(row=1, column=1, value='CREDIT — Uncertain-pixel analysis (AU / EU / TU)').font = Font(
        name='Arial', bold=True, size=14)

    first = next(iter(pixel_uncertainty.values()))
    au_t, eu_t, tu_t = first['thresholds']
    notes = [
        f'Thresholds used: AU > {au_t}, EU > {eu_t}, TU > {tu_t}.',
        'TU is computed as TU = AU + EU in this pipeline, so the identity holds for every pixel (see "Identity check").',
        'Because EU >= 0, TU >= AU for every pixel. Pixel COUNTS can still differ per criterion because each uses its own threshold.',
        'AU is an entropy in nats (max = ln #classes); EU is a mean of sigmoid outputs in [0, 1]. Equal thresholds are not directly comparable.',
        'Row / Col / Pixel_ID are 1-based. Reference_Class 0 = unlabelled pixel. Pixel sheets list pixels uncertain under AU, EU or TU (sorted by TU).',
    ]
    r = 3
    for line in notes:
        wa.cell(row=r, column=1, value=line).font = _BODY_FONT
        r += 1
    r += 1

    for mname, res in pixel_uncertainty.items():
        a = res['analysis']
        wa.cell(row=r, column=1, value=f'{mname}').font = Font(name='Arial', bold=True, size=13, color='1F4E79')
        r += 1
        wa.cell(row=r, column=1,
                value=f"Scene pixels: {a['n_scene_pixels']:,}   |   Uncertain under AU, EU or TU: {res['n_uncertain_total']:,}"
                      + ('   |   ⚠️ pixel sheet truncated to Excel row limit (see CSV)' if res['truncated'] else '')
                ).font = _BODY_FONT
        r += 2
        r = _write_df_to_sheet(wa, a['counts'],      start_row=r, title='Pixels flagged at the chosen thresholds') + 1
        r = _write_df_to_sheet(wa, a['overlap'],     start_row=r, title='Overlap between AU, EU and TU masks') + 1
        r = _write_df_to_sheet(wa, a['au_vs_tu'],    start_row=r, title='AU vs TU') + 1
        r = _write_df_to_sheet(wa, a['identity'],    start_row=r, title='Identity check  AU + EU = TU') + 1
        r = _write_df_to_sheet(wa, a['sensitivity'], start_row=r, title='Threshold sensitivity (same threshold t applied to AU, EU and TU)') + 2

    wa.column_dimensions['A'].width = 44
    for col in 'BCDEFGHIJ':
        wa.column_dimensions[col].width = 20


## 8.3 — Run Spatial Mapping for All Models

In [26]:
scene_uncertainty = {}   # per-model AU / EU / TU arrays for the Excel pixel sheet

for model_name, builder in model_builders.items():
    best_path = MODEL_DIR / f'{model_name}_CREDIT_best.keras'
    if not best_path.exists():
        print(f'[{model_name}] No weights found — skipping spatial maps.')
        continue

    tf.keras.backend.clear_session()
    student = build_credit_student(builder, num_classes)
    student.load_weights(best_path)

    scene_uncertainty[model_name] = generate_spatial_credit_maps(
        model_name, student, scene_patches, H, W,
        au_thresh=0.5, eu_thresh=0.2, tu_thresh=0.7
    )

  Inferring full scene for AlexNet_CNN...


2026-11-04 05:12:57.939267: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng13{} for conv (f32[2048,384,9,9]{3,2,1,0}, u8[0]{0}) custom-call(f32[2048,256,9,9]{3,2,1,0}, f32[384,256,3,3]{3,2,1,0}, f32[384]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActivationForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kRelu","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-11-04 05:12:58.027012: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.087905434s
Trying algorithm eng13{} for conv (f32[2048,384,9,9]{3,2,1,0}, u8[0]{0}) custom-call(f32[2048,256,9,9]{3,2,1,0}, f32[384,256,3,3]{3,2,1,0}, f32[384]{0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBiasActiv

  Saved → /kaggle/working/credit/results/AlexNet_CNN_CREDIT_spatial_maps.png
  Inferring full scene for GFNet...
  Saved → /kaggle/working/credit/results/GFNet_CREDIT_spatial_maps.png
  Inferring full scene for ViT_UNet...
  Saved → /kaggle/working/credit/results/ViT_UNet_CREDIT_spatial_maps.png


# 9.0 — Excel Report Export

Collects all saved plot paths (confusion matrices, spatial uncertainty maps, and any training-curve PNGs) and writes the final `CREDIT_Results.xlsx` workbook.

In [27]:
# ── Collect all plot paths ─────────────────────────────────────────────────────
_plot_entries = []

# Confusion matrix image
_cm_path = str(RESULTS_DIR / 'credit_confusion_matrices.png')
_plot_entries.append(('Confusion Matrices (all models)', _cm_path))

# Per-model spatial uncertainty maps
for _mname in model_builders:
    _sp_path = str(RESULTS_DIR / f'{_mname}_CREDIT_spatial_maps.png')
    _plot_entries.append((f'{_mname} — Spatial Uncertainty Maps', _sp_path))

# Any additional training-curve PNGs saved to PLOT_DIR
for _p in sorted(PLOT_DIR.glob('*.png')):
    _plot_entries.append((_p.stem.replace('_', ' ').title(), str(_p)))

# ── Pixel-level AU / EU / TU tables + threshold analysis (one sheet per model) ──
print('Building uncertain-pixel tables...')
_pixel_uncertainty = {
    _m: prepare_pixel_uncertainty(_m, _unc, y_img, RESULTS_DIR)
    for _m, _unc in scene_uncertainty.items()
}

# ── Write the workbook ─────────────────────────────────────────────────────────
create_credit_excel_report(
    results_dir           = RESULTS_DIR,
    train_summary_df      = _credit_train_summary_df,
    eval_summary_df       = _credit_eval_df,
    credit_artifacts_dict = _credit_artifacts_dict,
    plot_paths            = _plot_entries,
    pixel_uncertainty     = _pixel_uncertainty,
    credal_overall_df     = _credal_overall_df,
    credal_classwise_df   = _credal_classwise_df,
)

Building uncertain-pixel tables...
  [AlexNet_CNN] uncertain pixels (AU|EU|TU): 11,803  → full table: /kaggle/working/credit/results/AlexNet_CNN_uncertain_pixels.csv
  [GFNet] uncertain pixels (AU|EU|TU): 8,613  → full table: /kaggle/working/credit/results/GFNet_uncertain_pixels.csv
  [ViT_UNet] uncertain pixels (AU|EU|TU): 13,161  → full table: /kaggle/working/credit/results/ViT_UNet_uncertain_pixels.csv

✅ Excel report saved → /kaggle/working/credit/results/CREDIT_Results.xlsx


'/kaggle/working/credit/results/CREDIT_Results.xlsx'

In [28]:
# ── Final output manifest ──────────────────────────────────────────────────────
import subprocess
print('\n=== CREDIT Output Files ===')
result = subprocess.run(
    ['find', '/kaggle/working/credit', '-type', 'f'],
    capture_output=True, text=True
)
for f in sorted(result.stdout.strip().split('\n')):
    if f:
        sz = Path(f).stat().st_size / 1024 / 1024
        print(f'  {f}  ({sz:.2f} MB)')
print('\n✅ CREDIT pipeline complete!')


=== CREDIT Output Files ===
  /kaggle/working/credit/models/AlexNet_CNN_CREDIT_best.keras  (259.54 MB)
  /kaggle/working/credit/models/GFNet_CREDIT_best.keras  (121.47 MB)
  /kaggle/working/credit/models/ViT_UNet_CREDIT_best.keras  (185.11 MB)
  /kaggle/working/credit/results/AlexNet_CNN_CREDIT_spatial_maps.png  (0.94 MB)
  /kaggle/working/credit/results/AlexNet_CNN_uncertain_pixels.csv  (0.61 MB)
  /kaggle/working/credit/results/CREDIT_Results.xlsx  (3.63 MB)
  /kaggle/working/credit/results/GFNet_CREDIT_spatial_maps.png  (0.91 MB)
  /kaggle/working/credit/results/GFNet_uncertain_pixels.csv  (0.44 MB)
  /kaggle/working/credit/results/ViT_UNet_CREDIT_spatial_maps.png  (0.95 MB)
  /kaggle/working/credit/results/ViT_UNet_uncertain_pixels.csv  (0.69 MB)
  /kaggle/working/credit/results/credit_confusion_matrices.png  (0.11 MB)
  /kaggle/working/credit/results/credit_credal_coverage_classwise.csv  (0.00 MB)
  /kaggle/working/credit/results/credit_credal_coverage_overall.csv  (0.00 MB)
  /k